<a href="https://colab.research.google.com/github/jumafernandez/ANN-UNSL/blob/main/notebooks/version_4/notebook_05_evaluacion_semantica_ema_dialog2flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Notebook 05 — Evaluación semántica EMA Dialog2Flow con LLM

**Versión corregida: usa exclusivamente IDs de Google Drive como fuente de entrada.**

No busca embeddings en `data/version2.0`, `data/version_4` ni en ningún esquema de directorios previo.  
Todos los archivos de entrada se descargan desde los IDs definidos en el manifest a una carpeta local de ejecución.

Compara:

- `estatico`: `embeddings_dialog2flow.npy`
- `dinamico`: `accumulative_embeddings_dialog2flow.npy`
- `ema_alpha_0_1` ... `ema_alpha_0_9`

Pregunta experimental:

> ¿Algún valor de `alpha` en EMA mejora la recuperación semántico-funcional respecto del acumulativo uniforme anterior?

## 1️⃣ Instalación e imports

In [ ]:
!pip install -q openai gdown faiss-cpu pandas numpy tqdm scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 45.7 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import gc
import shutil
from pathlib import Path
from getpass import getpass
from datetime import datetime
from zoneinfo import ZoneInfo

import gdown
import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm
from IPython.display import display, Markdown

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)

Mounted at /content/drive
IN_COLAB: True


## 2️⃣ Configuración general

Los archivos de entrada se descargan por ID en:

```text
/content/ann_ema_d2f_inputs
```

Los resultados locales se guardan en:

```text
/content/ann_ema_d2f_results
```

Y si Google Drive está montado, se copian además a:

```text
/content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4
```

In [ ]:
# ============================================================
# Carpetas locales de trabajo
# ============================================================

INPUT_DIR = Path("/content/ann_ema_d2f_inputs")
RESULTS_DIR = Path("/content/ann_ema_d2f_results")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Sólo para copiar resultados, no para buscar entradas.
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4")
if IN_COLAB:
    DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Experimento
# ============================================================

EMBEDDING_NAME = "dialog2flow-joint-bert-base"
EMBEDDING_LABEL = "Dialog2Flow"
SHORT_NAME = "dialog2flow"

ALPHA_VALUES = [round(x / 10, 1) for x in range(1, 10)]

def alpha_tag(alpha: float) -> str:
    return f"{alpha:.1f}".replace(".", "_")

VARIANTES_A_EVALUAR = (
    ["estatico", "dinamico"] +
    [f"ema_alpha_{alpha_tag(alpha)}" for alpha in ALPHA_VALUES]
)

INDICES_A_EVALUAR = ["IVF", "HNSW", "IVFPQ"]

# Para prueba rápida:
# N_QUERIES_LLM = 10
# VARIANTES_A_EVALUAR = ["estatico", "dinamico", "ema_alpha_0_5"]
# INDICES_A_EVALUAR = ["HNSW"]

K_EVAL = 10
N_QUERIES_LLM = 100
N_QUERIES_SPLIT = 10000
RANDOM_STATE = 42
CONTEXT_WINDOW = 2

# ANN
N_LIST = 4096
N_PROBE = 64

HNSW_M = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 256

PQ_M = 32
PQ_NBITS = 8

N_TRAIN_IVF = 100_000

# OpenAI
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0
SLEEP_BETWEEN_CALLS = 0.2

print("INPUT_DIR:", INPUT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("Variantes:", VARIANTES_A_EVALUAR)
print("Índices:", INDICES_A_EVALUAR)
print("N_QUERIES_LLM:", N_QUERIES_LLM)

INPUT_DIR: /content/ann_ema_d2f_inputs
RESULTS_DIR: /content/ann_ema_d2f_results
Variantes: ['estatico', 'dinamico', 'ema_alpha_0_1', 'ema_alpha_0_2', 'ema_alpha_0_3', 'ema_alpha_0_4', 'ema_alpha_0_5', 'ema_alpha_0_6', 'ema_alpha_0_7', 'ema_alpha_0_8', 'ema_alpha_0_9']
Índices: ['IVF', 'HNSW', 'IVFPQ']
N_QUERIES_LLM: 100


## 3️⃣ Manifest de archivos por ID

Este es el único lugar donde se definen los archivos de entrada.

La función `download_file()` descarga **siempre por ID** al nombre local esperado.  
No busca en carpetas previas.

In [ ]:
FILES = {
    # Dataset e insumos
    "dialogs-2.0.pkl": "1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb",
    "ids.npy": "1hWC7nLvSboFHyykjr7VFAEmcvz8re1cY",

    # Dialog2Flow estático y acumulativo anterior
    "embeddings_dialog2flow.npy": "1Pxf6oho0HYwv3B8VObZ_R6asShyK1WFW",
    "accumulative_embeddings_dialog2flow.npy": "1ywEjHOKNRWkq7YbzmCzP-JT4DUgm6zAU",

    # EMA Dialog2Flow
    "ema_embeddings_dialog2flow_alpha_0_1.npy": "1na2SS7OxjMdHwzkKs_HPftbbKUV9NTdX",
    "ema_embeddings_dialog2flow_alpha_0_2.npy": "1j9F_P7CVd0EJ3TTI3cRtTb1bbIXoL4x1",
    "ema_embeddings_dialog2flow_alpha_0_3.npy": "1yqZJUVAo919r45wZWkkADiKdFdr44Sx8",
    "ema_embeddings_dialog2flow_alpha_0_4.npy": "1s9SVF6J1uBhgVWNSJnMqUzAqMjgETyw0",
    "ema_embeddings_dialog2flow_alpha_0_5.npy": "1fr5IJEi8vJaF17D74Rhog1MbieG-3Nt5",
    "ema_embeddings_dialog2flow_alpha_0_6.npy": "1oGqM0cpIkfGacPr3OY-DW-GYrlYMFH4h",
    "ema_embeddings_dialog2flow_alpha_0_7.npy": "1KmtA7MiAUgm2I7NRTy1Rxi9OZPUIGpqh",
    "ema_embeddings_dialog2flow_alpha_0_8.npy": "1UFR-nSi1J37IRVSe30GvxKWoW5bUULcN",
    "ema_embeddings_dialog2flow_alpha_0_9.npy": "1hN-AED1dO64hYDk_nccyqqXgy2VtCJZZ",
}


def local_path(filename: str) -> Path:
    return INPUT_DIR / filename


def download_file(filename: str, force: bool = False) -> Path:
    if filename not in FILES:
        raise KeyError(f"Archivo no registrado en FILES: {filename}")

    path = local_path(filename)

    if path.exists() and not force:
        return path

    if path.exists() and force:
        path.unlink()

    file_id = FILES[filename]
    url = f"https://drive.google.com/uc?id={file_id}"

    print("=" * 80)
    print("Descargando por ID")
    print("Archivo:", filename)
    print("ID:", file_id)
    print("Destino:", path)

    gdown.download(url, str(path), quiet=False, fuzzy=True)

    if not path.exists():
        raise FileNotFoundError(f"No se pudo descargar {filename} en {path}")

    return path


def variant_to_filename(variante: str) -> str:
    if variante == "estatico":
        return "embeddings_dialog2flow.npy"

    if variante == "dinamico":
        return "accumulative_embeddings_dialog2flow.npy"

    if variante.startswith("ema_alpha_"):
        tag = variante.replace("ema_alpha_", "")
        return f"ema_embeddings_dialog2flow_alpha_{tag}.npy"

    raise ValueError(f"Variante no reconocida: {variante}")


def get_embedding_path(variante: str) -> Path:
    return download_file(variant_to_filename(variante), force=False)

## 4️⃣ Descarga mínima inicial y carga del dataset

Primero se descargan sólo `dialogs-2.0.pkl` e `ids.npy`.

Los embeddings se descargan después, a demanda, al construir cada índice.

In [ ]:
ruta_df = download_file("dialogs-2.0.pkl")
ruta_ids = download_file("ids.npy")

df = pd.read_pickle(ruta_df).reset_index(drop=True)
ids = np.load(ruta_ids)

print("DataFrame:", df.shape)
print("IDs:", ids.shape)
display(df.head())

if len(df) != len(ids):
    print("ADVERTENCIA: len(df) != len(ids)")

Descargando por ID
Archivo: dialogs-2.0.pkl
ID: 1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb
Destino: /content/ann_ema_d2f_inputs/dialogs-2.0.pkl


Downloading...
From (original): https://drive.google.com/uc?id=1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb
From (redirected): https://drive.google.com/uc?id=1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb&confirm=t&uuid=48d37cb0-7a50-47c2-8874-7050894a3940
To: /content/ann_ema_d2f_inputs/dialogs-2.0.pkl
100%|██████████| 156M/156M [00:01<00:00, 84.4MB/s]


Descargando por ID
Archivo: ids.npy
ID: 1hWC7nLvSboFHyykjr7VFAEmcvz8re1cY
Destino: /content/ann_ema_d2f_inputs/ids.npy


Downloading...
From: https://drive.google.com/uc?id=1hWC7nLvSboFHyykjr7VFAEmcvz8re1cY
To: /content/ann_ema_d2f_inputs/ids.npy
100%|██████████| 8.00M/8.00M [00:00<00:00, 70.9MB/s]


DataFrame: (1000023, 11)
IDs: (1000023,)


,dataset,split,dialogue_id,turn_id,speaker,utterance,domains,dialog_acts,main_acts,slots,intents
0,ABCD,test,ABCD_test_0,0,user,Hi. My name is Chloe Zhang. I am curious as ...,[storewide query],None,None,None,[timing]
1,ABCD,test,ABCD_test_1,0,user,Hello. I recently signed up for a subscription...,[subscription inquiry],None,None,None,[manage dispute bill]
2,ABCD,test,ABCD_test_1,1,user,"sure, it's Albert Sanders and my account id is...",[subscription inquiry],None,None,[account id],None
3,ABCD,test,ABCD_test_1,2,user,yes its 7149958247,[subscription inquiry],None,None,[order id],None
4,ABCD,test,ABCD_test_2,0,user,I'm thinking about buying an item but first i ...,[single item query],None,None,None,[shirt]


## 5️⃣ Diagnóstico de embeddings

Esta celda descarga/verifica los embeddings necesarios.  
Si preferís no descargar todo al inicio, podés saltearla: los archivos se descargarán a demanda más adelante.

In [ ]:
RUN_FULL_DOWNLOAD_CHECK = True

if RUN_FULL_DOWNLOAD_CHECK:
    archivos_necesarios = [
        "embeddings_dialog2flow.npy",
        "accumulative_embeddings_dialog2flow.npy",
    ] + [
        variant_to_filename(v)
        for v in VARIANTES_A_EVALUAR
        if v.startswith("ema_alpha_")
    ]

    for filename in archivos_necesarios:
        path = download_file(filename, force=False)
        arr = np.load(path, mmap_mode="r")
        size_gb = path.stat().st_size / (1024**3)
        print(f"OK {filename:45s} | {size_gb:6.2f} GB | shape={arr.shape} | dtype={arr.dtype}")

        if arr.shape[0] != len(df):
            print("  ADVERTENCIA: cantidad de filas distinta del dataframe")
else:
    print("RUN_FULL_DOWNLOAD_CHECK=False. Se omite verificación completa.")

Descargando por ID
Archivo: embeddings_dialog2flow.npy
ID: 1Pxf6oho0HYwv3B8VObZ_R6asShyK1WFW
Destino: /content/ann_ema_d2f_inputs/embeddings_dialog2flow.npy


Downloading...
From (original): https://drive.google.com/uc?id=1Pxf6oho0HYwv3B8VObZ_R6asShyK1WFW
From (redirected): https://drive.google.com/uc?id=1Pxf6oho0HYwv3B8VObZ_R6asShyK1WFW&confirm=t&uuid=3efb4888-df32-41e1-8d71-1bf68c244101
To: /content/ann_ema_d2f_inputs/embeddings_dialog2flow.npy
100%|██████████| 3.07G/3.07G [00:41<00:00, 74.4MB/s]


OK embeddings_dialog2flow.npy                    |   2.86 GB | shape=(1000023, 768) | dtype=float32
Descargando por ID
Archivo: accumulative_embeddings_dialog2flow.npy
ID: 1ywEjHOKNRWkq7YbzmCzP-JT4DUgm6zAU
Destino: /content/ann_ema_d2f_inputs/accumulative_embeddings_dialog2flow.npy


Downloading...
From (original): https://drive.google.com/uc?id=1ywEjHOKNRWkq7YbzmCzP-JT4DUgm6zAU
From (redirected): https://drive.google.com/uc?id=1ywEjHOKNRWkq7YbzmCzP-JT4DUgm6zAU&confirm=t&uuid=37104cdf-b20a-43e5-89ba-31bcfd341506
To: /content/ann_ema_d2f_inputs/accumulative_embeddings_dialog2flow.npy
100%|██████████| 3.07G/3.07G [00:20<00:00, 151MB/s]


OK accumulative_embeddings_dialog2flow.npy       |   2.86 GB | shape=(1000023, 768) | dtype=float32
Descargando por ID
Archivo: ema_embeddings_dialog2flow_alpha_0_1.npy
ID: 1na2SS7OxjMdHwzkKs_HPftbbKUV9NTdX
Destino: /content/ann_ema_d2f_inputs/ema_embeddings_dialog2flow_alpha_0_1.npy


...

OK ema_embeddings_dialog2flow_alpha_0_7.npy      |   2.86 GB | shape=(1000023, 768) | dtype=float32
Descargando por ID
Archivo: ema_embeddings_dialog2flow_alpha_0_8.npy
ID: 1UFR-nSi1J37IRVSe30GvxKWoW5bUULcN
Destino: /content/ann_ema_d2f_inputs/ema_embeddings_dialog2flow_alpha_0_8.npy


Downloading...
From (original): https://drive.google.com/uc?id=1UFR-nSi1J37IRVSe30GvxKWoW5bUULcN
From (redirected): https://drive.google.com/uc?id=1UFR-nSi1J37IRVSe30GvxKWoW5bUULcN&confirm=t&uuid=3efcc809-9783-45f1-8635-ad357743c60e
To: /content/ann_ema_d2f_inputs/ema_embeddings_dialog2flow_alpha_0_8.npy
100%|██████████| 3.07G/3.07G [00:25<00:00, 119MB/s]


OK ema_embeddings_dialog2flow_alpha_0_8.npy      |   2.86 GB | shape=(1000023, 768) | dtype=float32
Descargando por ID
Archivo: ema_embeddings_dialog2flow_alpha_0_9.npy
ID: 1hN-AED1dO64hYDk_nccyqqXgy2VtCJZZ
Destino: /content/ann_ema_d2f_inputs/ema_embeddings_dialog2flow_alpha_0_9.npy


Downloading...
From (original): https://drive.google.com/uc?id=1hN-AED1dO64hYDk_nccyqqXgy2VtCJZZ
From (redirected): https://drive.google.com/uc?id=1hN-AED1dO64hYDk_nccyqqXgy2VtCJZZ&confirm=t&uuid=e78ac611-1a78-43ff-bfbc-94ff09064e6b
To: /content/ann_ema_d2f_inputs/ema_embeddings_dialog2flow_alpha_0_9.npy
100%|██████████| 3.07G/3.07G [00:41<00:00, 74.7MB/s]

OK ema_embeddings_dialog2flow_alpha_0_9.npy      |   2.86 GB | shape=(1000023, 768) | dtype=float32


## 6️⃣ Contexto conversacional para el juez

In [ ]:
_df_ordered = df.reset_index().rename(columns={"index": "row_id"})

_dialogue_groups = {
    dialogue_id: g.sort_values("turn_id")["row_id"].tolist()
    for dialogue_id, g in _df_ordered.groupby("dialogue_id", sort=False)
}

_row_to_pos = {}
for dialogue_id, rows in _dialogue_groups.items():
    for pos, row_id in enumerate(rows):
        _row_to_pos[int(row_id)] = int(pos)


def get_context_rows(row_id: int, window: int = 2):
    dialogue_id = df.loc[row_id, "dialogue_id"]
    rows = _dialogue_groups[dialogue_id]
    pos = _row_to_pos[int(row_id)]
    start = max(0, pos - window)
    return rows[start:pos + 1]


def format_turn(row_id: int) -> str:
    r = df.loc[row_id]
    speaker = r["speaker"] if "speaker" in df.columns else "?"
    utterance = str(r["utterance"]) if "utterance" in df.columns else str(r.to_dict())
    turn_id = r["turn_id"] if "turn_id" in df.columns else row_id
    return f"[{turn_id}] {speaker}: {utterance}"


def format_situation(row_id: int, window: int = 2) -> str:
    rows = get_context_rows(row_id, window=window)
    return "\n".join(format_turn(rid) for rid in rows)


print(format_situation(0, window=CONTEXT_WINDOW))

[0] user: Hi.  My name is Chloe Zhang.  I am curious as to when my promo code expires. Would you be able to tell me?


## 7️⃣ Split de consultas

Se genera localmente con `RANDOM_STATE=42`.  
Las consultas se excluyen del índice, igual que en las notebooks anteriores.

In [ ]:
def preparar_split(n_total: int, n_queries: int, random_state: int):
    rng = np.random.default_rng(random_state)
    query_idx = rng.choice(n_total, size=n_queries, replace=False)
    mask = np.ones(n_total, dtype=bool)
    mask[query_idx] = False
    index_idx = np.where(mask)[0]
    return index_idx.astype("int64"), query_idx.astype("int64")


split_index_path = INPUT_DIR / f"index_idx_N{N_QUERIES_SPLIT}_seed{RANDOM_STATE}.npy"
split_query_path = INPUT_DIR / f"query_idx_N{N_QUERIES_SPLIT}_seed{RANDOM_STATE}.npy"

if split_index_path.exists() and split_query_path.exists():
    index_idx = np.load(split_index_path)
    query_idx_full = np.load(split_query_path)
    print("Split local cargado.")
else:
    index_idx, query_idx_full = preparar_split(len(df), N_QUERIES_SPLIT, RANDOM_STATE)
    np.save(split_index_path, index_idx)
    np.save(split_query_path, query_idx_full)
    print("Split generado y guardado localmente.")

rng = np.random.default_rng(RANDOM_STATE + 100)
query_sample = rng.choice(
    query_idx_full,
    size=min(N_QUERIES_LLM, len(query_idx_full)),
    replace=False
).astype("int64")

print("Vectores indexados:", len(index_idx))
print("Consultas split completo:", len(query_idx_full))
print("Consultas para LLM:", len(query_sample))
print("Primeras consultas:", query_sample[:10])

Split generado y guardado localmente.
Vectores indexados: 990023
Consultas split completo: 10000
Consultas para LLM: 100
Primeras consultas: [258133 185989 734763 306913 846179 332360 169096 447851 940288 337047]


## 8️⃣ Recuperación top-10 con ANN

In [ ]:
def normalize_copy(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype="float32").copy()
    faiss.normalize_L2(x)
    return x


def build_ann_index(index_type: str, index_vectors: np.ndarray, random_state: int = 42):
    n, d = index_vectors.shape

    if index_type == "IVF":
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFFlat(quantizer, d, N_LIST, faiss.METRIC_L2)

        rng = np.random.default_rng(random_state)
        train_size = min(N_TRAIN_IVF, n)
        train_idx = rng.choice(n, size=train_size, replace=False)

        index.train(index_vectors[train_idx])
        index.add(index_vectors)
        index.nprobe = N_PROBE
        return index

    if index_type == "HNSW":
        index = faiss.IndexHNSWFlat(d, HNSW_M)
        index.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
        index.add(index_vectors)
        index.hnsw.efSearch = HNSW_EF_SEARCH
        return index

    if index_type == "IVFPQ":
        if d % PQ_M != 0:
            raise ValueError(f"La dimensión {d} no es divisible por PQ_M={PQ_M}")

        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFPQ(quantizer, d, N_LIST, PQ_M, PQ_NBITS)

        rng = np.random.default_rng(random_state)
        train_size = min(N_TRAIN_IVF, n)
        train_idx = rng.choice(n, size=train_size, replace=False)

        index.train(index_vectors[train_idx])
        index.add(index_vectors)
        index.nprobe = N_PROBE
        return index

    raise ValueError(f"Índice no reconocido: {index_type}")


def retrieve_topk_for_config(variante: str, index_type: str, k: int = 10):
    path = get_embedding_path(variante)
    matriz = np.load(path, mmap_mode="r")

    print("=" * 80)
    print("Variante:", variante, "| Índice:", index_type)
    print("Archivo:", path.name, "| Shape:", matriz.shape)

    index_vectors = normalize_copy(matriz[index_idx])
    query_vectors = normalize_copy(matriz[query_sample])

    t0 = time.time()
    index = build_ann_index(index_type, index_vectors, random_state=RANDOM_STATE)
    build_time = time.time() - t0

    t1 = time.time()
    distances, local_neighbors = index.search(query_vectors, k)
    search_time = time.time() - t1

    neighbor_rows = index_idx[local_neighbors]

    records = []
    for q_pos, q_row in enumerate(query_sample):
        q_row = int(q_row)

        for rank in range(k):
            n_row = int(neighbor_rows[q_pos, rank])

            records.append({
                "embedding_name": EMBEDDING_NAME,
                "embedding_base": EMBEDDING_LABEL,
                "variant": variante,
                "index_type": index_type,
                "query_order": int(q_pos),
                "query_row": q_row,
                "neighbor_rank": int(rank + 1),
                "neighbor_row": n_row,
                "distance": float(distances[q_pos, rank]),
                "query_utterance": str(df.loc[q_row, "utterance"]),
                "neighbor_utterance": str(df.loc[n_row, "utterance"]),
                "query_context": format_situation(q_row, window=CONTEXT_WINDOW),
                "neighbor_context": format_situation(n_row, window=CONTEXT_WINDOW),
                "build_time_s": build_time,
                "search_time_s": search_time,
            })

    del matriz, index_vectors, query_vectors, index, distances, local_neighbors, neighbor_rows
    gc.collect()

    return pd.DataFrame(records)


def run_all_retrievals():
    all_parts = []
    total = len(VARIANTES_A_EVALUAR) * len(INDICES_A_EVALUAR)
    done = 0

    for variante in VARIANTES_A_EVALUAR:
        for index_type in INDICES_A_EVALUAR:
            done += 1
            print(f"\n[{done}/{total}] {variante} / {index_type}")
            all_parts.append(retrieve_topk_for_config(variante, index_type, k=K_EVAL))

    return pd.concat(all_parts, ignore_index=True)


df_retrieval = run_all_retrievals()
print("Recuperaciones:", df_retrieval.shape)
display(df_retrieval.head())


[1/33] estatico / IVF
Variante: estatico | Índice: IVF
Archivo: embeddings_dialog2flow.npy | Shape: (1000023, 768)

[2/33] estatico / HNSW
Variante: estatico | Índice: HNSW
Archivo: embeddings_dialog2flow.npy | Shape: (1000023, 768)

[3/33] estatico / IVFPQ
Variante: estatico | Índice: IVFPQ
Archivo: embeddings_dialog2flow.npy | Shape: (1000023, 768)

[4/33] dinamico / IVF
Variante: dinamico | Índice: IVF
Archivo: accumulative_embeddings_dialog2flow.npy | Shape: (1000023, 768)

[5/33] dinamico / HNSW
Variante: dinamico | Índice: HNSW
Archivo: accumulative_embeddings_dialog2flow.npy | Shape: (1000023, 768)

[6/33] dinamico / IVFPQ
Variante: dinamico | Índice: IVFPQ
Archivo: accumulative_embeddings_dialog2flow.npy | Shape: (1000023, 768)

[7/33] ema_alpha_0_1 / IVF
Variante: ema_alpha_0_1 | Índice: IVF
Archivo: ema_embeddings_dialog2flow_alpha_0_1.npy | Shape: (1000023, 768)

[8/33] ema_alpha_0_1 / HNSW
Variante: ema_alpha_0_1 | Índice: HNSW
Archivo: ema_embeddings_dialog2flow_alpha_0_1

,embedding_name,embedding_base,variant,index_type,query_order,query_row,neighbor_rank,neighbor_row,distance,query_utterance,neighbor_utterance,query_context,neighbor_context,build_time_s,search_time_s
0,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,258133,1,572964,5.911047e-13,"oh , i completely forgot to mention that i wil...","Oh, I completely forgot to mention that I will...",[6] user: i 'd also like to find a 4 star hote...,[8] user: I'd also like to find a 4 star hotel...,233.845082,0.50986
1,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,258133,2,92687,5.996744e-13,"oh , i completely forgot to mention that i wil...","oh, i completely forgot to mention that i will...",[6] user: i 'd also like to find a 4 star hote...,[8] user: i'd also like to find a 4 star hotel...,233.845082,0.50986
2,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,258133,3,683102,1.183576e-12,"oh , i completely forgot to mention that i wil...","Oh, I completely forgot to mention that I will...",[6] user: i 'd also like to find a 4 star hote...,[3] user: The booking will be for only one per...,233.845082,0.50986
3,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,258133,4,116471,3.937679e-01,"oh , i completely forgot to mention that i wil...",oh i'm sorry i forgot to mention i must have a...,[6] user: i 'd also like to find a 4 star hote...,"[6] user: friday, for 1 person and will stay f...",233.845082,0.50986
4,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,258133,5,323472,3.937679e-01,"oh , i completely forgot to mention that i wil...",oh i 'm sorry i forgot to mention i must have ...,[6] user: i 'd also like to find a 4 star hote...,"[5] user: friday , for 1 person and will stay ...",233.845082,0.50986


## 9️⃣ Inspección rápida

In [ ]:
def mostrar_vecinos(
    df_retrieval: pd.DataFrame,
    variante: str = "ema_alpha_0_5",
    index_type: str = "HNSW",
    query_order: int = 0,
    top_n: int = 10,
):
    sub = df_retrieval[
        (df_retrieval["variant"] == variante) &
        (df_retrieval["index_type"] == index_type) &
        (df_retrieval["query_order"] == query_order)
    ].sort_values("neighbor_rank").head(top_n)

    if sub.empty:
        print("No hay registros para ese filtro.")
        return

    first = sub.iloc[0]
    display(Markdown(f"### Consulta `{query_order}` — {variante} / {index_type}"))
    display(Markdown("**Situación de consulta:**"))
    print(first["query_context"])

    display(sub[["neighbor_rank", "distance", "neighbor_row", "neighbor_utterance"]])


mostrar_vecinos(df_retrieval, variante="ema_alpha_0_5", index_type="HNSW", query_order=0)

### Consulta `0` — ema_alpha_0_5 / HNSW

**Situación de consulta:**

[6] user: i 'd also like to find a 4 star hotel with free wifi , please .
[7] system: there are 3 hotels which are 4 stars , two in the west and one in the centre . is there an area which you prefer ?
[8] user: oh , i completely forgot to mention that i will also need a hotel that has free parking . any of the 7 you said have this option ?


,neighbor_rank,distance,neighbor_row,neighbor_utterance
19000,1,0.031174,92687,"oh, i completely forgot to mention that i will..."
19001,2,0.031174,572964,"Oh, I completely forgot to mention that I will..."
19002,3,0.082570,258132,"there are 3 hotels which are 4 stars , two in ..."
19003,4,0.084549,258134,"huntingdon marriott hotel , the cambridge belf..."
19004,5,0.106730,572965,"huntingdon marriott hotel, the cambridge belfr..."
19005,6,0.106730,92688,"huntingdon marriott hotel, the cambridge belfr..."
19006,7,0.108717,92686,"there are 3 hotels which are 4 stars, two in t..."
19007,8,0.108717,572963,"There are 3 hotels which are 4 stars, two in t..."
19008,9,0.169209,258131,i 'd also like to find a 4 star hotel with fre...
19009,10,0.184514,324668,it should include free parking . the hotel sho...


## 🔟 OpenAI

In [ ]:
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Ingresá OPENAI_API_KEY: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("Cliente OpenAI configurado.")

Ingresá OPENAI_API_KEY: ··········
Cliente OpenAI configurado.


## 1️⃣1️⃣ Prompt juez

In [ ]:
EVALUATION_SCHEMA = {
    "name": "dialog_retrieval_evaluation",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "evaluations": {
                "type": "array",
                "minItems": K_EVAL,
                "maxItems": K_EVAL,
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "rank": {"type": "integer", "minimum": 1, "maximum": K_EVAL},
                        "semantic_similarity": {"type": "integer", "minimum": 1, "maximum": 5},
                        "functional_similarity": {"type": "integer", "minimum": 1, "maximum": 5},
                        "memory_usefulness": {"type": "integer", "minimum": 1, "maximum": 5},
                        "overall_similarity": {"type": "integer", "minimum": 1, "maximum": 5},
                        "brief_reason": {"type": "string"},
                    },
                    "required": [
                        "rank",
                        "semantic_similarity",
                        "functional_similarity",
                        "memory_usefulness",
                        "overall_similarity",
                        "brief_reason",
                    ],
                },
            }
        },
        "required": ["evaluations"],
    },
    "strict": True,
}

SYSTEM_PROMPT = """
You are an expert evaluator of task-oriented dialogue retrieval.
Your task is to judge whether retrieved dialogue situations are semantically and functionally similar to a query dialogue situation.
Focus on task-oriented dialogue behavior, not only lexical overlap.
Use the 1-5 scale consistently.
Return only valid JSON following the schema.
""".strip()


def build_judge_prompt(group: pd.DataFrame) -> str:
    group = group.sort_values("neighbor_rank")
    first = group.iloc[0]

    lines = []
    lines.append("Evaluate whether each retrieved neighbor is similar to the query situation.")
    lines.append("")
    lines.append("Scoring scale:")
    lines.append("1 = unrelated")
    lines.append("2 = weak or superficial relation")
    lines.append("3 = partial similarity")
    lines.append("4 = clear semantic/functional similarity")
    lines.append("5 = highly equivalent dialogue situations")
    lines.append("")
    lines.append("QUERY SITUATION:")
    lines.append(first["query_context"])
    lines.append("")
    lines.append("RETRIEVED NEIGHBORS:")

    for _, r in group.iterrows():
        lines.append("")
        lines.append(f"Neighbor rank {int(r['neighbor_rank'])}:")
        lines.append(r["neighbor_context"])

    lines.append("")
    lines.append(f"Return one evaluation object for each neighbor rank from 1 to {K_EVAL}.")
    return "\n".join(lines)


def evaluate_group_with_llm(group: pd.DataFrame, model: str = OPENAI_MODEL) -> dict:
    user_prompt = build_judge_prompt(group)

    response = client.chat.completions.create(
        model=model,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": EVALUATION_SCHEMA,
        },
    )

    return json.loads(response.choices[0].message.content)


_test_key, _test_group = next(iter(df_retrieval.groupby(["embedding_base", "variant", "index_type", "query_order"])))
print(build_judge_prompt(_test_group)[:2500])

Evaluate whether each retrieved neighbor is similar to the query situation.

Scoring scale:
1 = unrelated
2 = weak or superficial relation
3 = partial similarity
4 = clear semantic/functional similarity
5 = highly equivalent dialogue situations

QUERY SITUATION:
[6] user: i 'd also like to find a 4 star hotel with free wifi , please .
[7] system: there are 3 hotels which are 4 stars , two in the west and one in the centre . is there an area which you prefer ?
[8] user: oh , i completely forgot to mention that i will also need a hotel that has free parking . any of the 7 you said have this option ?

RETRIEVED NEIGHBORS:

Neighbor rank 1:
[8] user: i'd also like to find a 4 star hotel with free wifi, please.
[9] system: there are 3 hotels which are 4 stars, two in the west and one in the centre. is there an area which you prefer?
[10] user: oh, i completely forgot to mention that i will also need a hotel that has free parking. any of the 7 you said have this option?

Neighbor rank 2:
[8]

## 1️⃣2️⃣ Evaluación LLM con cache

In [ ]:
CACHE_PATH = RESULTS_DIR / "llm_semantic_scores_cache_ema_dialog2flow.jsonl"
print("Cache local:", CACHE_PATH)


def config_key_from_group(group: pd.DataFrame) -> str:
    first = group.iloc[0]
    return "|".join([
        str(first["embedding_base"]),
        str(first["variant"]),
        str(first["index_type"]),
        str(first["query_order"]),
        str(first["query_row"]),
        OPENAI_MODEL,
    ])


def load_existing_cache(path: Path) -> set:
    done = set()
    if not path.exists():
        return done

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                done.add(rec["config_key"])

    return done


def append_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def run_llm_evaluation(df_retrieval: pd.DataFrame, cache_path: Path):
    group_cols = ["embedding_base", "variant", "index_type", "query_order"]
    groups = list(df_retrieval.groupby(group_cols, sort=False))
    done = load_existing_cache(cache_path)

    print("Grupos a evaluar:", len(groups))
    print("Ya evaluados en cache:", len(done))

    for key, group in tqdm(groups):
        group = group.sort_values("neighbor_rank")
        config_key = config_key_from_group(group)

        if config_key in done:
            continue

        try:
            result = evaluate_group_with_llm(group, model=OPENAI_MODEL)

            record = {
                "config_key": config_key,
                "embedding_base": key[0],
                "variant": key[1],
                "index_type": key[2],
                "query_order": int(key[3]),
                "query_row": int(group.iloc[0]["query_row"]),
                "model": OPENAI_MODEL,
                "evaluations": result["evaluations"],
            }

            append_jsonl(cache_path, record)
            done.add(config_key)
            time.sleep(SLEEP_BETWEEN_CALLS)

        except Exception as e:
            print("Error en", key, "->", repr(e))
            time.sleep(2)

    print("Evaluación terminada.")


run_llm_evaluation(df_retrieval, CACHE_PATH)

NameError: name 'RESULTS_DIR' is not defined

In [ ]:
!ls

sample_data


## 1️⃣3️⃣ Cache a DataFrame

In [ ]:
def cache_to_dataframe(cache_path: Path, df_retrieval: pd.DataFrame) -> pd.DataFrame:
    records = []

    if not cache_path.exists():
        raise FileNotFoundError(cache_path)

    with open(cache_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            rec = json.loads(line)

            for ev in rec["evaluations"]:
                records.append({
                    "embedding_base": rec["embedding_base"],
                    "variant": rec["variant"],
                    "index_type": rec["index_type"],
                    "query_order": rec["query_order"],
                    "query_row": rec["query_row"],
                    "model": rec["model"],
                    **ev,
                })

    scores = pd.DataFrame(records)
    scores = scores.rename(columns={"rank": "neighbor_rank"})

    merge_cols = [
        "embedding_base", "variant", "index_type",
        "query_order", "query_row", "neighbor_rank"
    ]

    enriched = scores.merge(
        df_retrieval,
        on=merge_cols,
        how="left",
        validate="one_to_one",
    )

    return enriched


df_scores = cache_to_dataframe(CACHE_PATH, df_retrieval)
print(df_scores.shape)
display(df_scores.head())

## 1️⃣4️⃣ Métricas MSS@10

In [ ]:
query_metrics = (
    df_scores
    .groupby(["embedding_base", "variant", "index_type", "query_order"], as_index=False)
    .agg(
        semantic_score_at10=("semantic_similarity", "mean"),
        functional_score_at10=("functional_similarity", "mean"),
        memory_score_at10=("memory_usefulness", "mean"),
        mss_at10=("overall_similarity", "mean"),
    )
)

summary_metrics = (
    query_metrics
    .groupby(["embedding_base", "variant", "index_type"], as_index=False)
    .agg(
        n_queries=("query_order", "nunique"),
        mean_semantic_score_global_at10=("mss_at10", "mean"),
        sd_semantic_score_global_at10=("mss_at10", "std"),
        mean_semantic_score_at10=("semantic_score_at10", "mean"),
        sd_semantic_score_at10=("semantic_score_at10", "std"),
        mean_functional_score_at10=("functional_score_at10", "mean"),
        sd_functional_score_at10=("functional_score_at10", "std"),
        mean_memory_score_at10=("memory_score_at10", "mean"),
        sd_memory_score_at10=("memory_score_at10", "std"),
    )
)

summary_metrics["se_semantic_score_global_at10"] = (
    summary_metrics["sd_semantic_score_global_at10"] / np.sqrt(summary_metrics["n_queries"])
)

summary_metrics["ci95_semantic_score_global_at10"] = (
    1.96 * summary_metrics["se_semantic_score_global_at10"]
)

for col in summary_metrics.select_dtypes(include=["float"]).columns:
    summary_metrics[col] = summary_metrics[col].round(3)

display(summary_metrics)

## 1️⃣5️⃣ Comparación pareada vs dinámico anterior

In [ ]:
paired_scores = query_metrics.pivot_table(
    index=["embedding_base", "index_type", "query_order"],
    columns="variant",
    values="mss_at10",
).reset_index()

for baseline in ["estatico", "dinamico"]:
    if baseline not in paired_scores.columns:
        raise ValueError(f"Falta baseline {baseline}. Columnas: {paired_scores.columns.tolist()}")


def variant_to_alpha(variant):
    if str(variant).startswith("ema_alpha_"):
        return float(str(variant).replace("ema_alpha_", "").replace("_", "."))
    return np.nan


try:
    from scipy.stats import wilcoxon
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False


def wilcoxon_safe(deltas):
    deltas = pd.Series(deltas).dropna()
    if SCIPY_AVAILABLE and len(deltas) > 0 and not np.allclose(deltas, 0):
        try:
            stat, p_value = wilcoxon(deltas)
            return stat, p_value
        except Exception:
            return np.nan, np.nan
    return np.nan, np.nan


rows = []
EPS = 1e-9
ema_variants = [v for v in VARIANTES_A_EVALUAR if v.startswith("ema_alpha_")]

for (embedding_base, index_type), group in paired_scores.groupby(["embedding_base", "index_type"], sort=False):
    for variant in ["dinamico"] + ema_variants:
        if variant not in group.columns:
            continue

        delta_vs_static = group[variant] - group["estatico"]
        delta_vs_dynamic = group[variant] - group["dinamico"]

        stat_static, p_static = wilcoxon_safe(delta_vs_static)
        stat_dynamic, p_dynamic = wilcoxon_safe(delta_vs_dynamic)

        rows.append({
            "embedding_base": embedding_base,
            "index_type": index_type,
            "variant": variant,
            "alpha": variant_to_alpha(variant),
            "n_queries": int(group["query_order"].nunique()),

            "mean_static_mss_at10": float(group["estatico"].mean()),
            "mean_dynamic_mss_at10": float(group["dinamico"].mean()),
            "mean_variant_mss_at10": float(group[variant].mean()),

            "mean_delta_vs_static": float(delta_vs_static.mean()),
            "median_delta_vs_static": float(delta_vs_static.median()),
            "sd_delta_vs_static": float(delta_vs_static.std()),
            "pct_variant_better_than_static": float((delta_vs_static > EPS).mean()),
            "pct_static_better_than_variant": float((delta_vs_static < -EPS).mean()),
            "wilcoxon_stat_vs_static": stat_static,
            "wilcoxon_p_vs_static": p_static,

            "mean_delta_vs_dynamic": float(delta_vs_dynamic.mean()),
            "median_delta_vs_dynamic": float(delta_vs_dynamic.median()),
            "sd_delta_vs_dynamic": float(delta_vs_dynamic.std()),
            "pct_variant_better_than_dynamic": float((delta_vs_dynamic > EPS).mean()),
            "pct_dynamic_better_than_variant": float((delta_vs_dynamic < -EPS).mean()),
            "wilcoxon_stat_vs_dynamic": stat_dynamic,
            "wilcoxon_p_vs_dynamic": p_dynamic,
        })

comparison_summary = pd.DataFrame(rows)

for col in comparison_summary.select_dtypes(include=["float"]).columns:
    comparison_summary[col] = comparison_summary[col].round(4)

display(comparison_summary)

pivot_delta_vs_dynamic = comparison_summary.pivot_table(
    index="variant",
    columns="index_type",
    values="mean_delta_vs_dynamic",
)

pivot_pct_better_than_dynamic = comparison_summary.pivot_table(
    index="variant",
    columns="index_type",
    values="pct_variant_better_than_dynamic",
)

variant_order = ["dinamico"] + ema_variants
index_order = ["IVF", "HNSW", "IVFPQ"]

pivot_delta_vs_dynamic = pivot_delta_vs_dynamic.reindex(
    [v for v in variant_order if v in pivot_delta_vs_dynamic.index]
)

pivot_pct_better_than_dynamic = pivot_pct_better_than_dynamic.reindex(
    [v for v in variant_order if v in pivot_pct_better_than_dynamic.index]
)

pivot_delta_vs_dynamic = pivot_delta_vs_dynamic[
    [c for c in index_order if c in pivot_delta_vs_dynamic.columns]
]

pivot_pct_better_than_dynamic = pivot_pct_better_than_dynamic[
    [c for c in index_order if c in pivot_pct_better_than_dynamic.columns]
]

print("Delta medio vs dinámico anterior")
display(pivot_delta_vs_dynamic)

print("% queries donde variante supera dinámico anterior")
display(pivot_pct_better_than_dynamic)

## 1️⃣6️⃣ Tablas compactas

In [ ]:
pivot_mss_mean = summary_metrics.pivot_table(
    index="embedding_base",
    columns=["variant", "index_type"],
    values="mean_semantic_score_global_at10",
)

pivot_mss_sd = summary_metrics.pivot_table(
    index="embedding_base",
    columns=["variant", "index_type"],
    values="sd_semantic_score_global_at10",
)

ordered_cols = []
for variant in VARIANTES_A_EVALUAR:
    for idx in ["IVF", "HNSW", "IVFPQ"]:
        if (variant, idx) in pivot_mss_mean.columns:
            ordered_cols.append((variant, idx))

pivot_mss_mean = pivot_mss_mean[ordered_cols]
pivot_mss_sd = pivot_mss_sd[[c for c in ordered_cols if c in pivot_mss_sd.columns]]

pivot_mss_mean_sd = pivot_mss_mean.copy().astype(object)

for col in pivot_mss_mean.columns:
    for row_idx in pivot_mss_mean.index:
        mean_val = pivot_mss_mean.loc[row_idx, col]
        sd_val = pivot_mss_sd.loc[row_idx, col]

        if pd.isna(mean_val):
            pivot_mss_mean_sd.loc[row_idx, col] = ""
        else:
            pivot_mss_mean_sd.loc[row_idx, col] = f"{mean_val:.3f} ± {sd_val:.3f}"

summary_alpha = summary_metrics.copy()
summary_alpha["alpha"] = summary_alpha["variant"].map(variant_to_alpha)
summary_alpha = summary_alpha[summary_alpha["variant"].astype(str).str.startswith("ema_alpha_")]

pivot_alpha_mss = summary_alpha.pivot_table(
    index="alpha",
    columns="index_type",
    values="mean_semantic_score_global_at10",
).sort_index()

print("MSS@10 — media")
display(pivot_mss_mean)

print("MSS@10 — media ± SD")
display(pivot_mss_mean_sd)

print("EMA: alpha vs MSS@10")
display(pivot_alpha_mss)

## 1️⃣7️⃣ Ejemplo con scores

In [ ]:
def mostrar_vecinos_con_scores(
    df_scores: pd.DataFrame,
    variante: str = "ema_alpha_0_5",
    index_type: str = "HNSW",
    query_order: int = 0,
    top_n: int = 10,
):
    sub = df_scores[
        (df_scores["embedding_base"] == EMBEDDING_LABEL) &
        (df_scores["variant"] == variante) &
        (df_scores["index_type"] == index_type) &
        (df_scores["query_order"] == query_order)
    ].sort_values("neighbor_rank").head(top_n)

    if sub.empty:
        print("No hay registros para ese filtro.")
        return

    first = sub.iloc[0]
    display(Markdown(f"### Consulta `{query_order}` — {variante} / {index_type}"))
    display(Markdown("**Situación de consulta:**"))
    print(first["query_context"])

    cols = [
        "neighbor_rank",
        "distance",
        "overall_similarity",
        "semantic_similarity",
        "functional_similarity",
        "memory_usefulness",
        "neighbor_utterance",
        "brief_reason",
    ]

    display(sub[cols])


mostrar_vecinos_con_scores(
    df_scores,
    variante="ema_alpha_0_5",
    index_type="HNSW",
    query_order=0,
)

## 1️⃣8️⃣ Exportación

In [ ]:
fecha_hora_arg = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires")).strftime("%Y%m%d_%H%M%S_ARG")

retrieval_csv = RESULTS_DIR / f"retrieval_top10_textual_ema_dialog2flow_{fecha_hora_arg}.csv"
scores_csv = RESULTS_DIR / f"llm_semantic_scores_pairs_ema_dialog2flow_{fecha_hora_arg}.csv"
query_metrics_csv = RESULTS_DIR / f"llm_semantic_query_metrics_ema_dialog2flow_{fecha_hora_arg}.csv"
summary_csv = RESULTS_DIR / f"llm_semantic_summary_ema_dialog2flow_{fecha_hora_arg}.csv"
comparison_csv = RESULTS_DIR / f"ema_vs_dynamic_summary_{fecha_hora_arg}.csv"
pivot_delta_csv = RESULTS_DIR / f"table_delta_ema_vs_dynamic_pivot_{fecha_hora_arg}.csv"
pivot_pct_csv = RESULTS_DIR / f"table_pct_ema_better_than_dynamic_pivot_{fecha_hora_arg}.csv"
pivot_mean_csv = RESULTS_DIR / f"table_mss_at10_mean_pivot_ema_{fecha_hora_arg}.csv"
pivot_sd_csv = RESULTS_DIR / f"table_mss_at10_sd_pivot_ema_{fecha_hora_arg}.csv"
pivot_mean_sd_csv = RESULTS_DIR / f"table_mss_at10_mean_sd_pivot_ema_{fecha_hora_arg}.csv"
pivot_alpha_csv = RESULTS_DIR / f"table_alpha_mss_at10_ema_{fecha_hora_arg}.csv"
pivot_mean_sd_latex = RESULTS_DIR / f"table_mss_at10_mean_sd_pivot_ema_{fecha_hora_arg}.tex"

df_retrieval.to_csv(retrieval_csv, index=False)
df_scores.to_csv(scores_csv, index=False)
query_metrics.to_csv(query_metrics_csv, index=False)
summary_metrics.to_csv(summary_csv, index=False)
comparison_summary.to_csv(comparison_csv, index=False)
pivot_delta_vs_dynamic.to_csv(pivot_delta_csv)
pivot_pct_better_than_dynamic.to_csv(pivot_pct_csv)
pivot_mss_mean.to_csv(pivot_mean_csv)
pivot_mss_sd.to_csv(pivot_sd_csv)
pivot_mss_mean_sd.to_csv(pivot_mean_sd_csv)
pivot_alpha_mss.to_csv(pivot_alpha_csv)

with open(pivot_mean_sd_latex, "w", encoding="utf-8") as f:
    f.write(pivot_mss_mean_sd.to_latex())

files_to_copy = [
    retrieval_csv,
    scores_csv,
    query_metrics_csv,
    summary_csv,
    comparison_csv,
    pivot_delta_csv,
    pivot_pct_csv,
    pivot_mean_csv,
    pivot_sd_csv,
    pivot_mean_sd_csv,
    pivot_alpha_csv,
    pivot_mean_sd_latex,
    CACHE_PATH,
]

print("Archivos locales guardados:")
for p in files_to_copy:
    print("-", p)

if IN_COLAB:
    DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    for p in files_to_copy:
        shutil.copy(p, DRIVE_RESULTS_DIR / p.name)

    latest_map = {
        retrieval_csv: "retrieval_top10_textual_ema_dialog2flow_latest.csv",
        scores_csv: "llm_semantic_scores_pairs_ema_dialog2flow_latest.csv",
        query_metrics_csv: "llm_semantic_query_metrics_ema_dialog2flow_latest.csv",
        summary_csv: "llm_semantic_summary_ema_dialog2flow_latest.csv",
        comparison_csv: "ema_vs_dynamic_summary_latest.csv",
        pivot_delta_csv: "table_delta_ema_vs_dynamic_pivot_latest.csv",
        pivot_pct_csv: "table_pct_ema_better_than_dynamic_pivot_latest.csv",
        pivot_mean_csv: "table_mss_at10_mean_pivot_ema_latest.csv",
        pivot_sd_csv: "table_mss_at10_sd_pivot_ema_latest.csv",
        pivot_mean_sd_csv: "table_mss_at10_mean_sd_pivot_ema_latest.csv",
        pivot_alpha_csv: "table_alpha_mss_at10_ema_latest.csv",
    }

    for src, latest_name in latest_map.items():
        shutil.copy(src, DRIVE_RESULTS_DIR / latest_name)

    print("\nCopiados a Drive en:")
    print(DRIVE_RESULTS_DIR)